#Construcción de la capa Gold

In [0]:
%sql
-- Paso A: Agregación de Telemática
CREATE OR REPLACE TABLE gold.aggregated_telematics AS
SELECT 
    chassis_no,
    AVG(speed) AS avg_speed,
    MAX(speed) AS max_speed,
    AVG(latitude) AS avg_lat,
    AVG(longitude) AS avg_lon,
    COUNT(*) AS total_events
FROM silver.telematics_clean
GROUP BY chassis_no;

-- Paso B y C: Gran Tabla Maestra (Customer + Claim + Policy + Telematics)
CREATE OR REPLACE TABLE gold.customer_claim_policy_telematics AS
SELECT 
    c.claim_no,
    c.incident_ts,
    p.policy_id,
    cust.firstname,
    cust.lastname,
    cust.address,
    t.avg_speed,
    t.max_speed
FROM silver.claims_clean c
INNER JOIN silver.policies_clean p ON c.policy_id = p.policy_id
INNER JOIN silver.customers_clean cust ON p.customer_id = cust.customer_id
LEFT JOIN gold.aggregated_telematics t ON p.chassis_no = t.chassis_no;
